In [ ]:
import onnxruntime as ort
import numpy as np
import cv2
import os
import shutil
from pathlib import Path

# --- Configuration ---
MODEL_PATH = "../yolo_models/hand-quantized.onnx"
IMAGE_PATH = "../images/many_people_raising_hands.jpg"
RYZEN_AI_INSTALL_PATH = os.environ.get('RYZEN_AI_INSTALLATION_PATH', r"C:\Windows\System32\AMD")

# Setup Vitis AI execution provider config
config_file_path = "vaip_config.json" # Usually needed alongside
# We need to find the XCLBIN file. 
# Typical path: C:\Windows\System32\AMD\RyzenAI\xclbin\phoenix\1x4.xclbin (or similar)
# We will search for it if not explicitly set.
xclbin_path = None
search_paths = [
    os.path.join(RYZEN_AI_INSTALL_PATH, "RyzenAI", "xclbin", "phoenix", "1x4.xclbin"),
    os.path.join(RYZEN_AI_INSTALL_PATH, "voe-4.0-win_amd64", "xclbins", "phoenix", "1x4.xclbin"),
]

for p in search_paths:
    if os.path.exists(p):
        xclbin_path = p
        break

if xclbin_path is None:
    print("Warning: XCLBIN file not found automatically. Please set xclbin_path manually.")
else:
    print(f"Using XCLBIN: {xclbin_path}")

# Create a dummy config if not present (often needed by Vitis EP)
if not os.path.exists(config_file_path):
    with open(config_file_path, 'w') as f:
        f.write('{"vaip": {}}') # Minimal config

Using XCLBIN: C:\Program Files\RyzenAI\1.7.0\voe-4.0-win_amd64\xclbins\phoenix\1x4.xclbin


In [3]:
# --- Preprocessing Function (Letterbox) ---
def letterbox_image(image, target_size=(224, 224), padding_color=(114, 114, 114)):
    pad_left  = 0
    pad_top  = 0
    SCALE = 1
    # 1. Calculate scaling ratio
    h, w = image.shape[:2]
    target_h, target_w = target_size
    ratio = min(target_w / w, target_h / h)
    SCALE=ratio
    # print(f"Scale used: {ratio}")
    new_w, new_h = int(w * ratio), int(h * ratio)

    # print(f"New Width: {new_w} New Height: {new_h} ")

    if target_h>new_h:
        # print(f"Padding added to either height of image: {(target_h-new_h)//2}")
        pad_top = (target_h-new_h)/2
    if target_w>new_w:
        # print(f"Padding added to either width of image: {(target_w-new_w)/2} ")
        pad_left = (target_w-new_w)/2
    # 2. Resize image
    resized_image = cv2.resize(image, (new_w, new_h))

    # 3. Create canvas
    canvas = np.full((target_h, target_w, 3), padding_color, dtype=np.uint8)

    # 4. Center image
    top = (target_h - new_h) // 2
    left = (target_w - new_w) // 2
    canvas[top:top+new_h, left:left+new_w] = resized_image

    return [canvas, pad_left, pad_top, SCALE]

In [4]:
# --- Load & Preprocess Image ---
image = cv2.imread(IMAGE_PATH)
resized_image, pad_left, pad_top, scale = letterbox_image(image, target_size=(224, 224))

# Convert to RGB (OpenCV uses BGR by default)
rgb_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)

# Convert to float and Normalize to [0, 1]
float_image = rgb_image.astype(np.float32) / 255.0

# Flip channels to CHW and add batch dimension
flipped_image_array = np.transpose(float_image, (2,0,1))
formatted_image = flipped_image_array[np.newaxis, :]

print(f"Input shape: {formatted_image.shape}")

Input shape: (1, 3, 224, 224)


In [ ]:
# --- Initialize NPU Session ---

# Prepare provider options
provider_options = [{
         'target': 'X1',
         'xlnx_enable_py3_round': 0,
         'xclbin': xclbin_path,
     }]


print("Creating Inference Session with VitisAIExecutionProvider...")
try:
    session = ort.InferenceSession(
        MODEL_PATH, 
        providers=['VitisAIExecutionProvider'],
        provider_options=provider_options
    )
    print("Session created successfully!")
except Exception as e:
    print(f"Failed to create NPU session: {e}")
    # Fallback to CPU for debugging flow if NPU fails
    print("Falling back to CPUExecutionProvider...")
    session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])

input_name = session.get_inputs()[0].name

Creating Inference Session with VitisAIExecutionProvider...
Session created successfully!


In [10]:
# --- Run Inference ---
print("\nRunning inference...")
# Warmup (optional but good for NPU)
# session.run(None, {input_name: formatted_image}) 

outputs = session.run(None, {input_name: formatted_image})
raw_tensor = outputs[0]
print(f"Output shape: {raw_tensor.shape}")


Running inference...
Output shape: (1, 68, 1029)


In [11]:
# --- Post Processing (Same as Original) ---

# Remove batch dim and transpose [84, 8400] -> [8400, 84]
squeezed_array = np.squeeze(raw_tensor)
image_outputs = np.transpose(squeezed_array)

# Filter by confidence
column_index = 4
specific_criteria = 0.5 
mask = image_outputs[:, column_index] >= specific_criteria
high_confidence_cells = image_outputs[mask]

print(f"High confidence detections: {len(high_confidence_cells)}")

boxes = []
confidences = []
keypoints_list = []

for row in high_confidence_cells:
    # 1. Extract Bounding Box (Indices 0, 1, 2, 3)
    # The model gives [center_x, center_y, width, height]
    cx, cy, w, h = row[0:4]
    
    # 2. Convert to Top-Left format for OpenCV NMS
    x_min = cx - (w / 2)
    y_min = cy - (h / 2)
    
    # 3. Append to our lists
    boxes.append([float(x_min), float(y_min), float(w), float(h)])
    confidences.append(float(row[4]))
    
    # 5. Extract Raw Keypoints (Indices 5 onwards)
    keypoints_list.append(row[5:])

# NMS
score_threshold = 0.5
nms_threshold = 0.45
indices = cv2.dnn.NMSBoxes(boxes, confidences, score_threshold, nms_threshold)

final_results = []
if len(indices) > 0:
    for i in indices.flatten():
        final_results.append({
            "box": boxes[i],
            "score": confidences[i],
            "keypoints": keypoints_list[i]
        })

print(f"Final unique detections: {len(final_results)}")

High confidence detections: 25
Final unique detections: 4


In [12]:
# --- Visualization ---

# Reload original image for drawing
original_image = cv2.imread(IMAGE_PATH)

for result in final_results:
    box = result['box']
    kps_flat = result['keypoints']
    
    # Reshape Keypoints (21 points, 3 vals each: x,y,vis)
    # Note: Check if your model outputs 3 values per keypoint. Standard YOLO-Pose does.
    # If len(kps_flat) is 63 -> 21*3. If 42 -> 21*2.
    num_kps = len(kps_flat) // 3
    keypoints_2d = np.array(kps_flat).reshape(num_kps, 3)

    # Undo Scaling & Padding for Box
    # box is [x, y, w, h]
    orig_x = (box[0] - pad_left) / scale
    orig_y = (box[1] - pad_top) / scale
    orig_w = box[2] / scale
    orig_h = box[3] / scale
    
    # Convert to int for drawing
    x, y, w, h = int(orig_x), int(orig_y), int(orig_w), int(orig_h)

    # Draw Box
    cv2.rectangle(original_image, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Undo Scaling for Keypoints
    keypoints_2d[:, 0] = (keypoints_2d[:, 0] - pad_left) / scale
    keypoints_2d[:, 1] = (keypoints_2d[:, 1] - pad_top) / scale
    final_keypoints = keypoints_2d.astype(int)

    # Draw Keypoints
    for kp in final_keypoints:
        kx, ky, vis = kp
        cv2.circle(original_image, (kx, ky), 3, (0, 0, 255), -1)

cv2.imshow('NPU Result', original_image)
cv2.waitKey(0)
cv2.destroyAllWindows()